# CS 513 Final Project - NYPD Vehicle Stop Report Data

##### Imports

In [15]:
# Name: Samuel Preston and Dean Filippone
# CWID: xxx & xxx
# Assignment: Final Project
# Purpose: Analyze data regarding NYPD Traffic Stops
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

#models imports
# Naive Bayes
from sklearn.naive_bayes import GaussianNB, CategoricalNB

# ANN (Artificial Neural Network)
from sklearn.neural_network import MLPClassifier

# KNN (K-Nearest Neighbors)
from sklearn.neighbors import KNeighborsClassifier

# CART (Classification and Regression Trees)
from sklearn.tree import DecisionTreeClassifier

# Logistic Regression
from sklearn.linear_model import LogisticRegression

# Random Forest
from sklearn.ensemble import RandomForestClassifier

# SVM
from sklearn.svm import SVC

# Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier

# XGBoost
from xgboost import XGBClassifier

##### Reading Data

In [16]:
df = pd.read_csv('NYPD_Vehicle_Stop_Reports_20250430.csv')

## Data Manipulation & Cleaning

In [17]:
df_cleaned = df.copy()

df_cleaned = df_cleaned.rename(columns={
    'EVNT_KEY': 'event_id',
    'OCCUR_DT': 'stop_date',
    'OCCUR_TM': 'stop_time',
    'CMD_CD': 'command_code',
    'VEH_SEIZED_FLG': 'vehicle_seized',
    'VEH_SEARCHED_FLG': 'vehicle_searched',
    'VEH_SEARCH_CONSENT_FLG': 'search_consent',
    'VEH_CHECKPOINT_FLG': 'checkpoint_stop',
    'FORCE_USED_FLG': 'force_used',
    'ARREST_MADE_FLG': 'arrest_made',
    'SUMMON_ISSUED_FLG': 'summons_issued',
    'VEH_CATEGORY': 'vehicle_type',
    'RPTED_AGE': 'driver_age',
    'SEX_CD': 'driver_sex',
    'RACE_DESC': 'driver_race',
    'LATITUDE': 'latitude',
    'LONGITUDE': 'longitude',
    'X_COORD_CD': 'x_coord',
    'Y_COORD_CD': 'y_coord',
    'datetime': 'stop_datetime'
})

In [18]:
# Combine stop_date and stop_time into a datetime column
df_cleaned['stop_datetime'] = pd.to_datetime(df_cleaned['stop_date'] + ' ' + df_cleaned['stop_time'], errors='coerce')

# Extract parts of the date
df_cleaned['stop_dayofweek'] = df_cleaned['stop_datetime'].dt.day_name()
df_cleaned['stop_month'] = df_cleaned['stop_datetime'].dt.month
df_cleaned['stop_year'] = df_cleaned['stop_datetime'].dt.year

# Map months to seasons (Northern Hemisphere)
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df_cleaned['stop_season'] = df_cleaned['stop_month'].apply(get_season)

In [19]:
# Extract hour from stop_datetime
df_cleaned['stop_hour'] = df_cleaned['stop_datetime'].dt.hour

# Categorize time of day
def get_time_of_day(hour):
    if hour >= 0 and hour < 6:
        return 'Late Night'
    elif hour < 12:
        return 'Morning'
    elif hour < 18:
        return 'Afternoon'
    else:
        return 'Evening'

df_cleaned['stop_time_of_day'] = df_cleaned['stop_hour'].apply(get_time_of_day)

In [20]:
# Convert command_code to string type
df_cleaned['command_code'] = df_cleaned['command_code'].astype(str)

# Show unique values and type confirmation
df_cleaned['command_code'].unique()[:10], df_cleaned['command_code'].dtype

(array(['1', '5', '7', '9', '10', '13', '14', '18', '19', '20'],
       dtype=object),
 dtype('O'))

In [21]:
df_cleaned['search_consent'] = df_cleaned['search_consent'].replace({'(null)': None})
df_cleaned['search_consent'] = df_cleaned['search_consent'].map({
    'Y': 'CONSENTED',
    'N': 'DENIED'
}).fillna('UNKNOWN')